In [3]:
import numpy as np
import pandas as pd
import requests


In [7]:
url = (
    "https://ifis.iowafloodcenter.org/ifis/ws/meta/ifis.objects.php?type=4&fields=id,foreign_id,lat,lng,description,river,town,active"
)
print(f"Fetching IFC data from {url}...")
response = requests.get(url)
response.raise_for_status

stations_data = response.csv()
stations_df = pd.DataFrame(stations_data)

stations_df["lat"] = pd.to_numeric(stations_df["lat"], errors="coerce")
stations_df["lng"] = pd.to_numeric(stations_df["lng"], errors="coerce")
stations_df = stations_df.dropna(subset=["lat", "lng"])

episodes_df = pd.read_csv("NOAA_NW_Final.csv")

episodes_df = episodes_df.dropna(subset=["BEGIN_LAT","BEGIN_LON"])

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1,long1,lat2,lon2 = map(np.radians, [lat1,lon1,lat2,lon2])
    dlat = lat2-lat1
    dlon = lon2-lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

max_distance_km = 20.0
matching_station_ids = set()

for _, station in stations_df.iterrows():
    s_lat, s_lng = station["lat"], station ["lng"]

    distances = haversine(
        s_lat,
        s_lng,
        episodes_df["BEGIN_lAT"].values,
        episodes_df["BEGIN_LON"].values,
    )

    if np.any(distances <= max_distance_km):
        matching_station_ids.add(station["id"])

nearby_stations_df = stations_df[
    stations_df["id"].isin(matching_station_ids)
].copy()

print(f"Total stations fetched: {len(stations_df)}")
print(f"Stations within 20 km of flood episodes: {len(nearby_stations_df)}")

print(nearby_stations_df[["id", "town", "river", "lat", "lng"]].head())
nearby_stations_df.to_csv("Nearby_IFIS_Stations.csv", index=False)


Fetching IFC data from https://ifis.iowafloodcenter.org/ifis/ws/meta/ifis.objects.php?type=4&fields=id,foreign_id,lat,lng,description,river,town,active...


AttributeError: 'Response' object has no attribute 'csv'

In [29]:
import io
import numpy as np
import pandas as pd
import requests

# 1. Fetch stations metadata from the exact IFIS endpoint
url = "https://ifis.iowafloodcenter.org/ifis/ws/meta/ifis.objects.php?type=4&fields=id,foreign_id,lat,lng,description,river,town,active"

print("Fetching IFIS stations metadata...")
response = requests.get(url)
response.raise_for_status()

# Define the exact fields you requested in the URL order
field_names = [
    "id",
    "foreign_id",
    "lat",
    "lng",
    "description",
    "river",
    "town",
    "active",
]

# Read the pipe-delimited response with no header, assigning your field names directly
stations_df = pd.read_csv(
    io.StringIO(response.text),
    sep="|",
    header=None,
    names=field_names,
    comment="#",
    on_bad_lines="skip",
)

# Ensure lat and lng are numeric and clean NaNs
stations_df["lat"] = pd.to_numeric(stations_df["lat"], errors="coerce")
stations_df["lng"] = pd.to_numeric(stations_df["lng"], errors="coerce")
stations_df = stations_df.dropna(subset=["lat", "lng"])

print(f"Total valid stations loaded: {len(stations_df)}")

# 2. Load your storm episodes CSV
episodes_df = pd.read_csv("NOAA_NW_Final.csv")
episodes_df = episodes_df.dropna(subset=["BEGIN_LAT", "BEGIN_LON"])


# 3. Haversine distance function (in km)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


# 4. Filter stations within 20 km of any flood episode
max_distance_km = 20.0
matching_station_ids = set()

for _, station in stations_df.iterrows():
    s_lat, s_lng = station["lat"], station["lng"]
    distances = haversine(
        s_lat,
        s_lng,
        episodes_df["BEGIN_LAT"].values,
        episodes_df["BEGIN_LON"].values,
    )
    if np.any(distances <= max_distance_km):
        matching_station_ids.add(station["id"])

nearby_stations_df = stations_df[
    stations_df["id"].isin(matching_station_ids)
].copy()

print(f"Stations within 20 km of flood episodes: {len(nearby_stations_df)}")

nearby_stations_df.to_csv("Nearby_IFIS_Stations.csv", index=False)
print("Saved to Nearby_IFIS_Stations.csv successfully.")

Fetching IFIS stations metadata...
Total valid stations loaded: 280
Stations within 20 km of flood episodes: 23
Saved to Nearby_IFIS_Stations.csv successfully.


In [3]:
import os
import time
import pandas as pd
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

output_dir = "station_csvs"
os.makedirs(output_dir, exist_ok=True)

nearby_stations_df = pd.read_csv("Nearby_IFIS_Stations.csv")
id_column = (
    "foreign_id" if "foreign_id" in nearby_stations_df.columns else "id"
)
station_codes = nearby_stations_df[id_column].dropna().unique()

print(f"Found {len(station_codes)} unique stations to query.")

start_date = "20240602"
end_date = "20240702"
success_count = 0

for code in station_codes:
    url = f"https://hydroiowa.org/api/riversensor/{code}/data/{start_date}/{end_date}"
    print(f"Fetching data for station: {code}...")

    try:
        response = requests.get(url, timeout=15, verify=False)

        if response.status_code == 200:
            json_data = response.json()

            if isinstance(json_data, dict):
                # Extract station-level metadata
                obs_id = json_data.get("observatoryId", code)
                obs_name = json_data.get("observatoryName", "")
                gauge_elev = json_data.get("gaugeElevation", "")
                issued_time = json_data.get("issuedTime", "")

                # Extract the time-series records list under 'observed'
                records = json_data.get("observed", [])

                if records:
                    df_station = pd.DataFrame(records)

                    # Add metadata columns to every row for context
                    df_station["observatoryId"] = obs_id
                    df_station["observatoryName"] = obs_name
                    df_station["gaugeElevation"] = gauge_elev
                    df_station["issuedTime"] = issued_time

                    file_path = os.path.join(output_dir, f"{code}_data.csv")
                    df_station.to_csv(file_path, index=False)

                    print(
                        f"  -> Saved {len(df_station)} rows and columns {df_station.columns.tolist()} to {file_path}"
                    )
                    success_count += 1
                else:
                    print(f"  -> No 'observed' records found in JSON for {code}.")
            else:
                print(f"  -> Unexpected JSON format for {code}.")
        else:
            print(f"  -> Failed to fetch {code} (Status code: {response.status_code})")

    except Exception as e:
        print(f"  -> Error fetching {code}: {e}")

    time.sleep(0.2)

print(
    f"\nDone! Successfully processed and saved tabular CSV files for {success_count} stations in the '{output_dir}/' folder."
)

Found 23 unique stations to query.
Fetching data for station: MUDCREEK01...
  -> Saved 2881 rows and columns ['validTime', 'elevation', 'discharge', 'observatoryId', 'observatoryName', 'gaugeElevation', 'issuedTime'] to station_csvs\MUDCREEK01_data.csv
Fetching data for station: LTLSIOUX03...
  -> Saved 2881 rows and columns ['validTime', 'elevation', 'discharge', 'observatoryId', 'observatoryName', 'gaugeElevation', 'issuedTime'] to station_csvs\LTLSIOUX03_data.csv
Fetching data for station: LTLSIOUX05...
  -> Saved 2630 rows and columns ['validTime', 'elevation', 'discharge', 'observatoryId', 'observatoryName', 'gaugeElevation', 'issuedTime'] to station_csvs\LTLSIOUX05_data.csv
Fetching data for station: LTLROCK01...
  -> Saved 2881 rows and columns ['validTime', 'elevation', 'discharge', 'observatoryId', 'observatoryName', 'gaugeElevation', 'issuedTime'] to station_csvs\LTLROCK01_data.csv
Fetching data for station: WFKLTSIOUX01...
  -> Saved 2625 rows and columns ['validTime', 'elev

In [7]:
import io
import os
import time
import numpy as np
import pandas as pd
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# 1. Fetch metadata for hydrostations using type=15
url = "https://ifis.iowafloodcenter.org/ifis/ws/meta/ifis.objects.php?type=15&fields=id,foreign_id,lat,lng,description,town,active"

print("Fetching hydrostations metadata...")
response = requests.get(url, verify=False, timeout=15)
response.raise_for_status()

field_names = ["id", "foreign_id", "lat", "lng", "description", "town", "active"]
hydro_stations_df = pd.read_csv(
    io.StringIO(response.text),
    sep="|",
    header=None,
    names=field_names,
    comment="#",
    on_bad_lines="skip",
)

hydro_stations_df["lat"] = pd.to_numeric(hydro_stations_df["lat"], errors="coerce")
hydro_stations_df["lng"] = pd.to_numeric(hydro_stations_df["lng"], errors="coerce")
hydro_stations_df = hydro_stations_df.dropna(subset=["lat", "lng"])

print(f"Total valid hydrostations loaded: {len(hydro_stations_df)}")

# 2. Filter stations within 20 km of your NOAA flood episodes
episodes_df = pd.read_csv("NOAA_NW_Final.csv")
episodes_df = episodes_df.dropna(subset=["BEGIN_LAT", "BEGIN_LON"])


def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


max_distance_km = 20.0
matching_station_ids = set()

for _, station in hydro_stations_df.iterrows():
    s_lat, s_lng = station["lat"], station["lng"]
    distances = haversine(
        s_lat,
        s_lng,
        episodes_df["BEGIN_LAT"].values,
        episodes_df["BEGIN_LON"].values,
    )
    if np.any(distances <= max_distance_km):
        matching_station_ids.add(station["id"])

nearby_hydro_df = hydro_stations_df[
    hydro_stations_df["id"].isin(matching_station_ids)
].copy()
nearby_hydro_df.to_csv("Nearby_Hydro_Stations.csv", index=False)
print(f"Hydrostations within 20 km: {len(nearby_hydro_df)}")

# 3. Pull time-series data using both foreign_id and id to handle API variations
output_dir = "hydro_station_csvs"
os.makedirs(output_dir, exist_ok=True)

start_date = "20240602"
end_date = "20260702"
success_count = 0

for _, row in nearby_hydro_df.iterrows():
    foreign_code = str(row["foreign_id"]).strip() if pd.notna(row["foreign_id"]) else ""
    internal_id = str(row["id"]).strip() if pd.notna(row["id"]) else ""

    # Try candidate codes (foreign_id first, then internal id if different)
    codes_to_try = [c for c in [foreign_code, internal_id] if c]

    success = False
    for code in codes_to_try:
        # Check both endpoint structures used by hydroiowa / IFIS
        endpoints = [
            f"https://hydroiowa.org/api/riversensor/{code}/data/{start_date}/{end_date}",
            f"https://ifis.iowafloodcenter.org/ifis/ws/data/sensor.php?site={code}&format=json&start={start_date}&end={end_date}",
        ]

        for ts_url in endpoints:
            print(
                f"Fetching data for station ID {internal_id} (code: {code}) from {ts_url}..."
            )
            try:
                ts_response = requests.get(ts_url, timeout=15, verify=False)
                if ts_response.status_code == 200:
                    json_data = ts_response.json()

                    # Handle dictionary response with 'observed' list
                    if isinstance(json_data, dict):
                        obs_id = json_data.get("observatoryId", code)
                        obs_name = json_data.get("observatoryName", "")
                        gauge_elev = json_data.get("gaugeElevation", "")
                        issued_time = json_data.get("issuedTime", "")

                        records = json_data.get("observed", [])
                        if not records:
                            # Check alternative keys
                            for k in ["data", "rows", "results", "records", "values"]:
                                if k in json_data and isinstance(json_data[k], list):
                                    records = json_data[k]
                                    break

                        if records:
                            df_station = pd.DataFrame(records)
                            df_station["observatoryId"] = obs_id
                            df_station["observatoryName"] = obs_name
                            df_station["gaugeElevation"] = gauge_elev
                            df_station["issuedTime"] = issued_time

                            file_path = os.path.join(
                                output_dir, f"{internal_id}_hydro_data.csv"
                            )
                            df_station.to_csv(file_path, index=False)
                            print(
                                f"  -> Saved {len(df_station)} rows and columns {df_station.columns.tolist()} to {file_path}"
                            )
                            success = True
                            break

                    elif isinstance(json_data, list) and json_data:
                        df_station = pd.DataFrame(json_data)
                        file_path = os.path.join(
                            output_dir, f"{internal_id}_hydro_data.csv"
                        )
                        df_station.to_csv(file_path, index=False)
                        print(
                            f"  -> Saved {len(df_station)} rows to {file_path}"
                        )
                        success = True
                        break
            except Exception as e:
                print(f"  -> Attempt failed: {e}")

        if success:
            success_count += 1
            break
        else:
            print(f"  -> No data retrieved for station {internal_id} using code {code}.")

    time.sleep(0.2)

print(
    f"\nDone! Successfully processed and saved individual hydrostation CSV files for {success_count} stations in the '{output_dir}/' folder."
)

Fetching hydrostations metadata...
Total valid hydrostations loaded: 59
Hydrostations within 20 km: 1
Fetching data for station ID 3021 (code: RACCOON04) from https://hydroiowa.org/api/riversensor/RACCOON04/data/20240602/20260702...
Fetching data for station ID 3021 (code: RACCOON04) from https://ifis.iowafloodcenter.org/ifis/ws/data/sensor.php?site=RACCOON04&format=json&start=20240602&end=20260702...
  -> No data retrieved for station 3021 using code RACCOON04.
Fetching data for station ID 3021 (code: 3021) from https://hydroiowa.org/api/riversensor/3021/data/20240602/20260702...
Fetching data for station ID 3021 (code: 3021) from https://ifis.iowafloodcenter.org/ifis/ws/data/sensor.php?site=3021&format=json&start=20240602&end=20260702...
  -> No data retrieved for station 3021 using code 3021.

Done! Successfully processed and saved individual hydrostation CSV files for 0 stations in the 'hydro_station_csvs/' folder.


In [1]:
import geopandas as gpd

In [ ]:
import io
import os
import time
import numpy as np
import pandas as pd
import requests
import urllib3
from shapely.geometry import LineString

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# 1. Fetch metadata for hydrostations using type=15
url = "https://ifis.iowafloodcenter.org/ifis/ws/meta/ifis.objects.php?type=15&fields=id,foreign_id,lat,lng,description,town,active"

print("Fetching hydrostations metadata...")
response = requests.get(url, verify=False, timeout=15)
response.raise_for_status()

field_names = ["id", "foreign_id", "lat", "lng", "description", "town", "active"]
hydro_stations_df = pd.read_csv(
    io.StringIO(response.text),
    sep="|",
    header=None,
    names=field_names,
    comment="#",
    on_bad_lines="skip",
)

hydro_stations_df["lat"] = pd.to_numeric(hydro_stations_df["lat"], errors="coerce")
hydro_stations_df["lng"] = pd.to_numeric(hydro_stations_df["lng"], errors="coerce")
hydro_stations_df = hydro_stations_df.dropna(subset=["lat", "lng"])

print(f"Total valid hydrostations loaded: {len(hydro_stations_df)}")

hs_gdf = gpd.GeoDataFrame(
    hydro_stations_df,
    geometry=gpd.points_from_xy(hydro_stations_df['lng'], hydro_stations_df['lat']),
    crs="EPSG:4326"
)

hs_gdf.to_file("hs.geojson", driver="GeoJSON")

print(f'Hydrostations successfully converted to GeoJSON!')

stormDF = pd.read_csv("NOAA_NW_Final.csv")

lines = [
    LineString([(s_lon, s_lat), (e_lon,)])
]


Fetching hydrostations metadata...
Total valid hydrostations loaded: 59
Hydrostations successfully converted to GeoJSON!


In [ ]:
sensors = gpd.read_file